In [ ]:

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip


builder = (
    SparkSession.builder
    .master("local[1]")
    .appName("Cross_System_Monitoring")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.python.worker.reuse", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

reading data

In [2]:
missing = spark.read.format("delta").load("../gold/missing_records")

duplicates = spark.read.format("delta").load("../gold/duplicates")

drift = spark.read.format("delta").load("../gold/drift_report")

count

In [3]:
missing_count = missing.count()

duplicate_count = duplicates.count()

drift_count = drift.filter(drift.difference > 100).count()

load drift

In [4]:
drift = spark.read.format("delta").load("../gold/drift_report")

dashboard score

In [5]:

score = 100.0

score -= (missing_count * 0.005)

score -= (duplicate_count * 0.01)

score -= (drift_count * 0.05)

score = max(round(score, 1), 0.0)

print(f"Dashboard Trust Score = {score}/100")

Dashboard Trust Score = 36.6/100


saving trust score

In [ ]:
trust = spark.createDataFrame([(score,)], ["trust_score"])

trust.write.format("delta").mode("overwrite").save("../gold/trust_score")